# **Vanilla DejaVu offload**

Cells copied unchanged (outputs kept) from `archive/OS_MAIN.ipynb`: original cell indices [0, 1, 5, 6, 7, 8, 9, 10, 11].

In [1]:
import torch
import time
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
import transformers

transformers.utils.logging.disable_progress_bar()

torch.cuda.set_per_process_memory_fraction(0.45)

# model_name = "gpt2"
model_name = "facebook/opt-6.7b"

device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    attn_implementation="eager"
).to(device)
model.eval()

print(f"Model: {model_name}")
print(f"Device: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

Model: facebook/opt-6.7b
Device: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB


In [2]:
from datasets import load_dataset

# WikiText-2
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

# Get long articles by filtering for substantial text
long_prompts = []
for item in dataset:
    text = item["text"].strip()
    tokens = tokenizer.encode(text)
    if len(tokens) > 200:  # only keep long articles
        long_prompts.append(text)
    if len(long_prompts) >= 32:
        break

print(f"Found {len(long_prompts)}  prompts")
for i, p in enumerate(long_prompts[:4]):
    tokens = tokenizer.encode(p)
    print(f"Prompt {i}: {len(tokens)} tokens | {p[:80]}...")

Found 32  prompts
Prompt 0: 217 tokens | He had a recurring role in 2003 on two episodes of The Bill , as character " Con...
Prompt 1: 205 tokens | In 2006 Boulter starred in the play Citizenship written by Mark Ravenhill . The ...
Prompt 2: 246 tokens | Boulter starred in two films in 2008 , Daylight Robbery by filmmaker Paris Leont...
Prompt 3: 268 tokens | Du Fu 's work is notable above all for its range . Chinese critics traditionally...


# **DEJAVU**

**Helper Functions**

In [2]:
def get_kv_layers(kv_cache):
    """Convert DynamicCache or other cache formats into standard ((k,v),(k,v),...) format."""
    if kv_cache is None:
        return None
    # Newer transformers: DynamicCache stores tensors in .layers[i].keys / .layers[i].values
    if hasattr(kv_cache, "layers") and isinstance(kv_cache.layers, list):
        return [(layer.keys, layer.values) for layer in kv_cache.layers]
    # Older transformers: DynamicCache stores tensors in .key_cache / .value_cache
    if hasattr(kv_cache, "key_cache") and hasattr(kv_cache, "value_cache"):
        return list(zip(kv_cache.key_cache, kv_cache.value_cache))
    # Try legacy tuple conversion as final fallback
    if hasattr(kv_cache, "to_legacy_cache"):
        legacy = kv_cache.to_legacy_cache()
        if isinstance(legacy, (tuple, list)):
            return legacy
    if isinstance(kv_cache, (tuple, list)):
        return kv_cache
    raise TypeError(f"Unsupported KV cache type: {type(kv_cache)}")

def get_cache_seq_len(kv_cache):
    """Return current sequence length of KV cache."""
    layers = get_kv_layers(kv_cache)
    if layers is None or len(layers) == 0:
        return 0
    first_key = layers[0][0]
    return first_key.shape[-2]

def move_full_kv_to_device_timed(kv_cache, target_device):
    """Move KV tensors from CPU → device and reconstruct a DynamicCache the model can consume."""
    start = time.perf_counter()
    dev_layers = [(k.to(target_device), v.to(target_device)) for k, v in kv_cache]
    # Rebuild as DynamicCache so the model receives the correct cache type
    cache = build_dynamic_cache(dev_layers)
    end = time.perf_counter()
    return cache, (end - start)

def extract_new_kv_chunk(full_kv_cache, old_seq_len):
    """Extract only newly generated KV entries from updated cache."""
    layers = get_kv_layers(full_kv_cache)
    new_layers = []
    for k, v in layers:
        new_k = k[:, :, old_seq_len:, :].detach().cpu()
        new_v = v[:, :, old_seq_len:, :].detach().cpu()
        new_layers.append((new_k, new_v))
    return tuple(new_layers)

def build_dynamic_cache(kv_layers):
  """Reconstruct a DynamicCache object from a list of (key, value) tensor pairs."""
  cache = DynamicCache()
  for layer_idx, (k, v) in enumerate(kv_layers):
    # update() expects shape (batch, heads, seq, head_dim) and stores it internally
    cache.update(k, v, layer_idx)
  return cache

def append_kv_caches(cpu_kv_cache, new_chunk_cache):
  """Append new KV chunk to CPU side KV cache."""
  merged = []
  for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_cache, new_chunk_cache):
    merged_k = torch.cat([old_k, new_k], dim=-2)
    merged_v = torch.cat([old_v, new_v], dim=-2)
    merged.append((merged_k, merged_v))
  return tuple(merged)

**BENCHMARK FUNCTION**

In [4]:
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"Allocated: {torch.cuda.memory_allocated() / (1024**2):.1f} MB")
print(f"Reserved: {torch.cuda.memory_reserved() / (1024**2):.1f} MB")

Allocated: 12700.0 MB
Reserved: 12704.0 MB


In [5]:
def run_dejavu_benchmark():
  microbatches_count = [8,16,32]
  n_tokens = [256,512]
  # microbatches_count = [2]
  # n_tokens = [32]
  results = []

  total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

  for D in microbatches_count:
    for N in n_tokens:

      prompts = long_prompts[:D] # Select D prompts

      torch.cuda.empty_cache()
      torch.cuda.reset_peak_memory_stats()

      total_start= time.perf_counter()

      with torch.no_grad():

        # PREFILL
        cpu_kv_caches = []
        next_tokens = []
        for i in range(D):
            inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
            outputs = model(input_ids=inputs.input_ids, use_cache=True)
            next_tokens.append(torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1))
            layers = get_kv_layers(outputs.past_key_values)
            cpu_kv_caches.append([(k.detach().cpu(), v.detach().cpu()) for k, v in layers])
            del outputs
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        transfer_time = 0.0
        # DECODE
        for step in range(N-1):
          for i in range(D):
            # Transfer KV cache from CPU to GPU
            gpu_cache, t_upload = move_full_kv_to_device_timed(cpu_kv_caches[i], device)
            old_seq_length = get_cache_seq_len(gpu_cache)
            outputs = model(
                input_ids = next_tokens[i],
                past_key_values = gpu_cache,
                use_cache = True
              )
            new_kv_chunk = extract_new_kv_chunk(outputs.past_key_values, old_seq_length)

            # Append newly generated kv chunk to the cpu cache
            t_start = time.perf_counter()
            cpu_kv_caches[i] = append_kv_caches(cpu_kv_caches[i], new_kv_chunk)
            t_stop = time.perf_counter()
            t_offload = t_stop - t_start

            next_tokens[i] = torch.argmax(outputs.logits[:,-1,:],dim=1).unsqueeze(1)

            transfer_time += t_upload
            transfer_time += t_offload

            torch.cuda.empty_cache()

        del gpu_cache, new_kv_chunk
        torch.cuda.empty_cache()
      total_end = time.perf_counter()

      # METRICS
      latency = total_end - total_start
      total_tokens = D * N
      throughput = total_tokens / latency
      peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
      vram_util = (peak_vram_mb / total_vram_mb) * 100
      transfer_overhead = (transfer_time / latency) * 100

      total_kv_bytes = 0
      for cache in cpu_kv_caches:
        for k, v in cache:
          total_kv_bytes += k.nelement() * k.element_size()
          total_kv_bytes += v.nelement() * v.element_size()
      cpu_kv_cache_mb = total_kv_bytes / (1024 ** 2)

      print(f"D={D:<4} N={N:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | CPU KV Cache={cpu_kv_cache_mb:.1f}MB | Transfer Time={transfer_time:.2f}s | Transfer Overhead={transfer_overhead:.2f}%")

      results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "CPU KV Cache Total (MB)": cpu_kv_cache_mb,
                "Transfer Time (s)": transfer_time,
                "Transfer Overhead (%)": transfer_overhead,
            })

      # CLEANUP
      del next_tokens
      torch.cuda.empty_cache()

  return pd.DataFrame(results)



In [6]:
run_dejavu_benchmark()

D=4    N=128  | Latency=46.834s | Throughput=10.9 tok/s | Peak VRAM=13429.2MB | VRAM%=33.2% | CPU KV Cache=722.0MB | Transfer Time=28.67s | Transfer Overhead=61.21%
D=4    N=256  | Latency=102.338s | Throughput=10.0 tok/s | Peak VRAM=13620.5MB | VRAM%=33.7% | CPU KV Cache=978.0MB | Transfer Time=68.42s | Transfer Overhead=66.85%
D=4    N=512  | Latency=247.317s | Throughput=8.3 tok/s | Peak VRAM=14002.0MB | VRAM%=34.6% | CPU KV Cache=1490.0MB | Transfer Time=178.55s | Transfer Overhead=72.19%
D=8    N=128  | Latency=92.602s | Throughput=11.1 tok/s | Peak VRAM=13954.9MB | VRAM%=34.5% | CPU KV Cache=1541.5MB | Transfer Time=58.80s | Transfer Overhead=63.50%
D=8    N=256  | Latency=208.572s | Throughput=9.8 tok/s | Peak VRAM=13932.5MB | VRAM%=34.5% | CPU KV Cache=2053.5MB | Transfer Time=140.61s | Transfer Overhead=67.41%
D=8    N=512  | Latency=505.531s | Throughput=8.1 tok/s | Peak VRAM=14081.5MB | VRAM%=34.8% | CPU KV Cache=3077.5MB | Transfer Time=368.72s | Transfer Overhead=72.94%
D=

,Microbatches (D),Tokens Generated (N),Latency (s),Throughput (tok/s),Peak VRAM (MB),VRAM Util (%),CPU KV Cache Total (MB),Transfer Time (s),Transfer Overhead (%)
0,4,128,46.834213,10.932179,13429.205078,33.206599,722.0,28.668351,61.212411
1,4,256,102.337944,10.006064,13620.501953,33.679621,978.0,68.415749,66.852768
2,4,512,247.316695,8.280881,14002.017578,34.623001,1490.0,178.548222,72.194165
3,8,128,92.601572,11.058128,13954.894043,34.506478,1541.5,58.803929,63.502085
4,8,256,208.572330,9.819136,13932.542480,34.451209,2053.5,140.605463,67.413287
5,8,512,505.531031,8.102371,14081.518555,34.819584,3077.5,368.717882,72.936746
6,16,128,190.448582,10.753559,15000.038086,37.090821,3067.5,122.573825,64.360587
7,16,256,429.098457,9.545595,15092.327148,37.319026,4091.5,292.347714,68.130684
8,16,512,1046.636200,7.826979,15104.108398,37.348157,6139.5,767.803776,73.359184
